# Housing Dataset Kubeflow Demo

## Data exploration

Import the California Housing dataset

In [ ]:
import sklearn

In [ ]:
from sklearn.datasets import fetch_california_housing

# Fetch the dataset as a bunch object containing a DataFrame
california = fetch_california_housing(as_frame=True)

# Access the combined DataFrame (Features + Target)
df = california.frame

In [ ]:
df["HouseAge"].hist()

## Training models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

import pandas as pd

# Load the preprocessed dataset
df = pd.read_parquet(output_file)

# Define the target column for prediction
target_column = "MedHouseVal"

# Split the data into training and testing sets
train_x, test_x, train_y, test_y = train_test_split(
    df.drop(columns=[target_column]),
    df[target_column],
    test_size=0.25,
    random_state=42
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

model.fit(train_x, train_y)

score = model.score(test_x, test_y)

print(f"Model R² Score: {score:.4f}")

## Use MLFlow

In [ ]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

import pandas as pd

# Define the target column for prediction
target_column = "MedHouseVal"

# Split the data into training and testing sets
train_x, test_x, train_y, test_y = train_test_split(
    df.drop(columns=[target_column]),
    df[target_column],
    test_size=0.25,
    random_state=42
)

# Enable MLflow auto logging for scikit-learn models
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="my-run") as run:
    mlflow.set_tag("author", "kf-testing")

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ])
    
    model.fit(train_x, train_y)

    score = model.score(test_x, test_y)

    mlflow.log_metric("test_r2", score)
    
    model_uri = f"{run.info.artifact_uri}/model"
    print(model_uri)

    mlflow.sklearn.log_model(model, "model", registered_model_name="my-test-model")

# Using Katib

In [ ]:
from kubeflow.katib import KatibClient
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# 1. Define your training function
def train_fn(parameters):
    # Parameters are passed as a dictionary
    n_estimators = int(parameters["n_estimators"])
    
    # Load and split
    data = fetch_california_housing()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    # Define and train pipeline
    model = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=n_estimators))
    model.fit(X_train, y_train)
    
    # Get metric
    r2 = model.score(X_test, y_test)
    
    # Important: The function must print the metric for Katib to capture it
    print(f"R2={r2}")

# 2. Initialize Katib Client
client = KatibClient()

# 3. Configure and run the experiment
client.tune(
    name="housing-estimators-tune",
    objective=train_fn,
    parameters={
        "n_estimators": "int(range: 50, 200)"  # Search space
    },
    objective_metric_name="R2",
    objective_type="maximize",
    max_trial_count=10,
    parallel_trial_count=2,
    base_image="python:3.11",
    packages_to_install=["scikit-learn", "pandas", "numpy"]
)

## Using KFP 

In [ ]:
@component(
    base_image="python:3.11",  # Use Python 3.11 base image
    packages_to_install=["pandas==2.3.3", "pyarrow==19.0.1"],
)
def create_dataset(output_file: OutputPath("Dataset")) -> None:
    from sklearn.datasets import fetch_california_housing

    # Fetch the dataset as a bunch object containing a DataFrame
    california = fetch_california_housing(as_frame=True)
    
    # Access the combined DataFrame (Features + Target)
    df = california.frame

    df.to_parquet(output_file)

@component(
    base_image="python:3.11",  # Use Python 3.11 base image
    packages_to_install=[
        "pandas==2.3.3",
        "scikit-learn==1.8.0",
        "mlflow==2.22.4",
        "pyarrow==19.0.1",
        "boto3==1.42.37",
    ],
)
def train_model(dataset: InputPath("Dataset"), run_name: str, model_name: str) -> str:
    import os
    import mlflow
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import r2_score
    
    import pandas as pd

    # Load the preprocessed dataset
    df = pd.read_parquet(dataset)

    # Define the target column for prediction
    target_column = "MedHouseVal"
    
    # Split the data into training and testing sets
    train_x, test_x, train_y, test_y = train_test_split(
        df.drop(columns=[target_column]),
        df[target_column],
        test_size=0.25,
        random_state=42
    )
    
    # Enable MLflow auto logging for scikit-learn models
    mlflow.sklearn.autolog()
    
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("author", "kf-testing")
    
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
        ])
        
        model.fit(train_x, train_y)
    
        score = model.score(test_x, test_y)
    
        mlflow.log_metric("test_r2", score)
        
        model_uri = f"{run.info.artifact_uri}/model"
        print(model_uri)
    
        mlflow.sklearn.log_model(model, "model", registered_model_name=model_name)


@component(
    base_image="python:3.11",  # Use Python 3.11 base image
    packages_to_install=["kserve==0.15.2", "kubernetes==30.1.0", "tenacity==9.1.2"],
)
def deploy_model_with_kserve(model_uri: str, isvc_name: str) -> str:
    from kubernetes.client import V1ObjectMeta
    from kserve import (
        constants,
        KServeClient,
        V1beta1InferenceService,
        V1beta1InferenceServiceSpec,
        V1beta1PredictorSpec,
        V1beta1SKLearnSpec,
    )
    from tenacity import retry, wait_exponential, stop_after_attempt

    # Initialize the Inference Service specification
    isvc = V1beta1InferenceService(
        api_version=constants.KSERVE_V1BETA1,
        kind=constants.KSERVE_KIND_INFERENCESERVICE,
        metadata=V1ObjectMeta(
            name=isvc_name,
            annotations={"sidecar.istio.io/inject": "false"},
        ),
        spec=V1beta1InferenceServiceSpec(
            predictor=V1beta1PredictorSpec(
                service_account_name="kserve-controller-s3",
                sklearn=V1beta1SKLearnSpec(storage_uri=model_uri),
            )
        ),
    )

    # Deploy the Inference Service using KServe
    client = KServeClient()
    client.create(isvc)

    # Retry logic to ensure the Inference Service is ready
    @retry(
        wait=wait_exponential(multiplier=2, min=1, max=10),
        stop=stop_after_attempt(30),
        reraise=True,
    )
    def assert_isvc_created(client, isvc_name):
        assert client.is_isvc_ready(isvc_name), f"Failed to create Inference Service {isvc_name}."

    # Wait until the service is ready and get the service URL
    assert_isvc_created(client, isvc_name)
    isvc_resp = client.get(isvc_name)
    isvc_url = isvc_resp["status"]["address"]["url"]
    print("Inference URL:", isvc_url)

    return isvc_url

putting all together

In [ ]:
# Define a constant for the Inference Service name
ISVC_NAME = "housing-regressor"
MLFLOW_RUN_NAME = "regressor-pipeline-xyz"
MLFLOW_MODEL_NAME = "regressor-pipeline"

# Fetch environment variables for MLflow tracking and AWS credentials
# These are guaranteed to be present because of the mlflow's poddefault please refer to [this guide](https://documentation.ubuntu.com/charmed-mlflow/en/latest/tutorial/mlflow-kubeflow/

mlflow_tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
mlflow_s3_endpoint_url = os.getenv("MLFLOW_S3_ENDPOINT_URL")
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")

@pipeline(name="download-preprocess-train-deploy-pipeline")
def download_preprocess_train_deploy_pipeline(url: str):
    # Step 1: Download the dataset from the URL
    download_task = download_dataset(url=url)

    # Step 2: Preprocess the downloaded dataset
    preprocess_task = preprocess_dataset(dataset=download_task.outputs["dataset_path"])

    # Step 3: Train the model on the preprocessed dataset
    train_task = (
        train_model(
            dataset=preprocess_task.outputs["output_file"],
            run_name=MLFLOW_RUN_NAME,
            model_name=MLFLOW_MODEL_NAME,
        )
        .set_env_variable(name="MLFLOW_TRACKING_URI", value=mlflow_tracking_uri)
        .set_env_variable(name="MLFLOW_S3_ENDPOINT_URL", value=mlflow_s3_endpoint_url)
        .set_env_variable(name="AWS_ACCESS_KEY_ID", value=aws_access_key_id)
        .set_env_variable(name="AWS_SECRET_ACCESS_KEY", value=aws_secret_access_key)
    )

    # Step 4: Deploy the trained model with KServe
    deploy_task = deploy_model_with_kserve(
        model_uri=train_task.output, isvc_name=ISVC_NAME
    ).set_env_variable(name="AWS_SECRET_ACCESS_KEY", value=aws_secret_access_key)

execute

In [ ]:
import kfp

client = kfp.Client()

kfp.compiler.Compiler().compile(
    download_preprocess_train_deploy_pipeline, "download_preprocess_train_deploy_pipeline.yaml"
)

run = client.create_run_from_pipeline_func(
    download_preprocess_train_deploy_pipeline, arguments={"url": url}, enable_caching=False
)

In [ ]:
import numpy as np

np.array(test_x.iloc[:2])

## Inference Service

In [ ]:
# Initialize the KServe client
# This client is used to interact with the KServe Inference Service.
kserve_client = KServeClient()

# Retrieve the Inference Service details
# Fetches the Inference Service by name and extracts the URL for making predictions.
isvc_resp = kserve_client.get(ISVC_NAME)
inference_service_url = isvc_resp["status"]["address"]["url"]
print("Inference URL:", inference_service_url)

# Define the input data for prediction
# This data matches the expected input format of the deployed model, with each instance being a list of feature values.
input_data = {
    "instances": [
        [ 1.68120000e+00,  2.50000000e+01,  4.19220056e+00, 1.02228412e+00,  1.39200000e+03,  3.87743733e+00, 3.60600000e+01, -1.19010000e+02],
        [ 2.53130000e+00,  3.00000000e+01,  5.03938356e+00, 1.19349315e+00,  1.56500000e+03,  2.67979452e+00, 3.51400000e+01, -1.19460000e+02]
    ]
}

# Send a prediction request to the Inference Service
# This sends the input data to the model for prediction via a POST request and prints the response.
response = requests.post(f"{inference_service_url}/v1/models/{ISVC_NAME}:predict", json=input_data)
print(response.text)